# Track 16 — Fine-Tuning versus RAG and Production Deployment
## The Architectural Convergence of Parametric and Non-Parametric Intelligence

### Curriculum Objective
In this concluding capstone track of the **RAG Engineering Academy**, we address the central strategic decision faced by AI system architects: **When should you retrieve (Non-Parametric Knowledge), when should you fine-tune (Parametric Knowledge), and how do you combine both into a hardened, production-ready serving architecture?**

We implement:
1. **The Multi-Dimensional Decision Matrix**: Evaluating trade-offs across Knowledge Currency, Citations, Style Adherence, Access Control, Latency, and Upfront Engineering Cost.
2. **Fine-Tuning Dataset Formats & Leakage Detection**: Schema validation and data leakage audits for Supervised Fine-Tuning (SFT) and Embedding Contrastive Triplet datasets.
3. **First-Principles Low-Rank Adaptation (LoRA)**: Mathematical decomposition of $\Delta W = \frac{\alpha}{r} (B \cdot A)$, weight merging for zero-overhead inference, and parameter efficiency accounting.
4. **CPU Quantization & Numerical Fidelity**: Symmetric and asymmetric INT8 quantization, scale/zero-point calibration, memory compression ($75\%$ reduction), and cosine fidelity verification.
5. **The Hybrid RAG + Fine-Tuning Synergy**: Combining dense non-parametric vector retrieval with fine-tuned structured synthesis and refusal gates against hallucinations.
6. **Production Serving Contract (FastAPI & ASGI)**: Enterprise-grade serving with Kubernetes health probes (`/health`), query endpoints with streaming latency headers, ingestion pipelines, and Prometheus telemetry.
7. **Containerization & Deployment Readiness Audit**: Automated 8-pillar compliance audit, multi-stage Dockerfile generation, and non-root security hardening.
8. **Academy Milestone Retrospective**: Connecting all 17 tracks into a unified engineering foundation.


### Environment Verification & Offline Configuration
We import the core production utilities from `common.finetuning_deployment` and ensure that all execution remains $100\%$ offline-executable, CPU-deterministic, and self-contained.


In [ ]:
import json
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Robust discovery of repository root across execution environments
cur = Path(".").resolve()
if (cur / "common").exists():
    REPO_ROOT = cur
elif (cur.parent / "common").exists():
    REPO_ROOT = cur.parent
elif (cur.parent.parent / "common").exists():
    REPO_ROOT = cur.parent.parent
else:
    REPO_ROOT = cur

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.finetuning_deployment import (
    ArchitectureRequirements,
    ArchitectureType,
    KnowledgeVolatility,
    evaluate_architecture_tradeoffs,
    DatasetValidator,
    LoRALinear,
    CPUQuantizer,
    HybridRAGFineTuningEngine,
    create_rag_production_app,
    DeploymentReadinessAuditor,
    generate_finetuning_vs_rag_diagram,
    generate_deployment_architecture_diagram,
)
from starlette.testclient import TestClient

print("All Track 16 production modules imported successfully!")
print(f"Workspace root: {REPO_ROOT}")


### Module 1: The Knowledge Spectrum — Parametric vs. Non-Parametric Memory

Every LLM application relies on two fundamentally different forms of memory:

| Dimension | Parametric Knowledge (Fine-Tuning / Weights) | Non-Parametric Knowledge (RAG / External Index) |
| :--- | :--- | :--- |
| **Storage Mechanism** | Continuous weights ($W$) modified via SGD | Discrete text chunks in vector DBs (HNSW, BM25) |
| **Update Velocity** | Slow & Expensive: requires re-training, eval, and deployment | Sub-second: insert/delete rows in vector index |
| **Hallucination Risk** | High: model confidently confabulates ungrounded facts | Low: grounded directly in retrieved context |
| **Exact Citations** | Incapable of reliable dynamic source offset citations | Native: returns exact chunk IDs, page numbers, URLs |
| **Style & Syntax Adherence** | **Superior**: internalizes grammar, tone, and JSON schemas | Weaker: relies on prompt engineering and few-shot examples |
| **Access Control (ACL)** | Impossible: weights cannot partition multi-tenant permissions | **Native**: metadata filters enforce strict user authorization |
| **Inference Cost & Latency** | Low context window overhead (compact prompt) | Higher latency due to retrieval + longer injected prompt |
| **Upfront Cost** | Higher: dataset annotation, GPU compute for tuning | Lower: straightforward embedding ingestion |

Let us evaluate the analytical decision engine on three representative enterprise archetypes:
1. **Dynamic Financial News Analyst**: High volatility, strict citations, low budget.
2. **Specialized Legal Contract Drafter**: Static legal terminology, custom syntax, large training set.
3. **Enterprise SOC2 Policy Assistant**: Moderate volatility, strict citations, and structured JSON schemas.


In [ ]:
scenarios = {
    "Dynamic Financial News": ArchitectureRequirements(
        knowledge_volatility=KnowledgeVolatility.REAL_TIME,
        need_strict_citations=True,
        custom_style_or_syntax=False,
        strict_json_schema=False,
        access_control_required=True,
        upfront_budget_usd=100.0,
        training_samples_available=50,
    ),
    "Specialized Legal Drafter": ArchitectureRequirements(
        knowledge_volatility=KnowledgeVolatility.STATIC,
        need_strict_citations=False,
        custom_style_or_syntax=True,
        strict_json_schema=True,
        access_control_required=False,
        upfront_budget_usd=800.0,
        training_samples_available=1500,
        latency_budget_ms=90.0,
    ),
    "Enterprise SOC2 Assistant": ArchitectureRequirements(
        knowledge_volatility=KnowledgeVolatility.DAILY_WEEKLY,
        need_strict_citations=True,
        custom_style_or_syntax=True,
        strict_json_schema=True,
        access_control_required=True,
        upfront_budget_usd=500.0,
        training_samples_available=300,
    ),
}

for name, req in scenarios.items():
    rec = evaluate_architecture_tradeoffs(req)
    print(f"=== Scenario: {name} ===")
    print(f"  Recommended Architecture : {rec.primary_architecture.value}")
    print(f"  Confidence Score         : {rec.confidence_score}")
    print(f"  Estimated Initial Cost   : ${rec.estimated_initial_cost_usd:.2f}")
    print("  Key Rationales:")
    for r in rec.rationales[:2]:
        print(f"    • {r}")
    print("  Serving Stack Recommendation:")
    for k, v in rec.recommended_serving_stack.items():
        print(f"    - {k}: {v}")
    print()


### Visualizing the Decision Quadrant and Trade-Off Profile
We render the publication-grade architectural decision matrix and dimensional radar bar profile. The quadrant divides problem spaces along two primary axes:
- **X-axis**: Knowledge Volatility & Citation Strictness (driving RAG adoption).
- **Y-axis**: Specialized Style & Output Syntax Complexity (driving Fine-Tuning adoption).


In [ ]:
diag_path = REPO_ROOT / "assets" / "diagrams" / "track16_finetuning_vs_rag_decision_matrix.png"
generate_finetuning_vs_rag_diagram(str(diag_path))

# Render inline
img = plt.imread(str(diag_path))
plt.figure(figsize=(15, 6.5))
plt.imshow(img)
plt.axis("off")
plt.title("Parametric vs. Non-Parametric Architectural Decision Matrix", fontsize=13, weight="bold", pad=10)
plt.tight_layout()
plt.close()
print(f"Decision matrix diagram loaded from: {diag_path}")


### Module 2: Fine-Tuning Dataset Engineering & Leakage Detection

Fine-tuning requires curated datasets conforming to strict schemas:
1. **Instruction SFT Datasets** (`instruction`, `input`, `output`): Teaches structured output schemas, extraction rules, and refusal handling.
2. **Contrastive Triplet Datasets** (`query`, `positive`, `negative`): Fine-tunes embedding bi-encoders using Multiple Negatives Ranking Loss (InfoNCE).

A critical step in dataset engineering is **data leakage detection**:
- **Target Leakage**: Output trivially duplicates the input without reasoning.
- **Contamination**: Duplicate instructions across splits leading to inflated test performance.
- **False Negative Triplets**: Positive and negative passages are identical or contradictory.

We inspect and validate our curated domain fixtures using `DatasetValidator`.


In [ ]:
# Load curated SFT fixture
sft_fixture_path = REPO_ROOT / "fixtures" / "finetuning" / "sample_sft_dataset.jsonl"
sft_samples = []
with open(sft_fixture_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            sft_samples.append(json.loads(line))

report_sft = DatasetValidator.validate_sft_dataset(sft_samples)
print("SFT Dataset Validation:")
print(f"  Valid Samples   : {report_sft.valid_count}")
print(f"  Invalid Samples : {report_sft.invalid_count}")
print(f"  Mean Input Tok  : {report_sft.mean_token_count_input}")
print(f"  Mean Output Tok : {report_sft.mean_token_count_output}")
print(f"  Leakages Found  : {len(report_sft.detected_leakages)}")

# Load curated Contrastive Triplets fixture
triplet_fixture_path = REPO_ROOT / "fixtures" / "finetuning" / "sample_contrastive_triplets.jsonl"
triplet_samples = []
with open(triplet_fixture_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            triplet_samples.append(json.loads(line))

report_triplet = DatasetValidator.validate_contrastive_triplets(triplet_samples)
print("\nContrastive Triplets Validation:")
print(f"  Valid Triplets  : {report_triplet.valid_count}")
print(f"  Mean Query Tok  : {report_triplet.mean_token_count_input}")
print(f"  Mean Doc Tok    : {report_triplet.mean_token_count_output}")
print(f"  Leakages Found  : {len(report_triplet.detected_leakages)}")


### Module 3: First-Principles Low-Rank Adaptation (LoRA)

Full parameter fine-tuning of an LLM requires updating and storing gigabytes of optimizer states for every weight matrix $W_0 \in \mathbb{R}^{d_{out} \times d_{in}}$.

**LoRA** freezes $W_0$ and decomposes the task update into two low-rank matrices:
$$\Delta W = \frac{\alpha}{r} (B \cdot A)$$
where:
- $A \in \mathbb{R}^{r \times d_{in}}$, initialized from $\mathcal{N}(0, \frac{1}{r})$
- $B \in \mathbb{R}^{d_{out} \times r}$, initialized to strictly zero ($B = 0$)
- $r \ll \min(d_{in}, d_{out})$ is the rank (typically $4, 8, 16$)
- $\alpha$ is a constant scaling hyperparameter

#### Zero Initial Overhead & Weight Merging
Because $B = 0$ at initialization, $\Delta W = 0$ on day zero. In production serving, the adapter weights can be **fused directly** into the base matrix:
$$W_{merged} = W_0 + \frac{\alpha}{r} (B \cdot A)$$
This eliminates all adapter latency overhead during deployment while retaining modularity.


In [ ]:
in_dim, out_dim, rank = 512, 512, 8
layer = LoRALinear(in_dim=in_dim, out_dim=out_dim, rank=rank, alpha=16.0, seed=42)

print(f"Base Parameters       : {layer.base_parameter_count:,} (FP32)")
print(f"LoRA Parameters       : {layer.lora_parameter_count:,} (Trainable)")
print(f"Parameter Efficiency  : {layer.parameter_reduction_ratio * 100:.2f}% of full weights!")

# Verify initial Delta W is exactly zero
delta_w_init = layer.compute_delta_w()
print(f"Initial Delta W Max Abs: {np.max(np.abs(delta_w_init)):.8f} (Exactly Zero!)")

# Simulate synthetic gradient step
target_delta = np.random.randn(out_dim, in_dim).astype(np.float32) * 0.05
mse = layer.train_step_mock(target_delta, lr=0.1)
print(f"Adapter Adaptation Step MSE: {mse:.6f}")

# Test forward pass before and after weight merging
x_test = np.random.randn(4, in_dim).astype(np.float32)
y_unmerged = layer.forward(x_test)

layer.merge_weights()
y_merged = layer.forward(x_test)

merge_diff = np.max(np.abs(y_unmerged - y_merged))
print(f"Weight Merging Invariance Max Diff: {merge_diff:.8e} (Identical Inference!)")


### Module 4: CPU Quantization & Numerical Fidelity (FP32 to INT8)

Deploying RAG embeddings and LLM projection heads on standard CPU infrastructure requires memory and cache optimization.

**Quantization** maps 32-bit floating point numbers (4 bytes) to 8-bit integers (1 byte), achieving a **75% memory reduction**:

1. **Symmetric INT8 Quantization**:
   $$S = \frac{\max(|x|)}{127}, \quad q = \text{clip}\left(\text{round}\left(\frac{x}{S}\right), -127, 127\right), \quad \hat{x} = q \cdot S$$
2. **Asymmetric INT8 Quantization**:
   $$S = \frac{x_{max} - x_{min}}{255}, \quad Z = \text{round}\left(-\frac{x_{min}}{S}\right), \quad \hat{x} = (q - Z) \cdot S$$

We evaluate the fidelity of INT8 compression on a weight matrix by calculating:
- **Cosine Fidelity**: Directional preservation ($\frac{\langle x, \hat{x} \rangle}{\|x\| \|\hat{x}\|}$)
- **Root Mean Square Error (RMSE)**
- **Signal-to-Noise Ratio (SNR dB)**


In [ ]:
rng = np.random.default_rng(123)
fp32_weights = rng.normal(0.0, 0.5, size=(256, 512)).astype(np.float32)

# Symmetric Quantization
q_sym = CPUQuantizer.quantize_symmetric_int8(fp32_weights)
recon_sym = CPUQuantizer.dequantize(q_sym)
metrics_sym = CPUQuantizer.compute_fidelity_metrics(fp32_weights, recon_sym)

print("Symmetric INT8 Quantization Fidelity:")
print(f"  Cosine Fidelity         : {metrics_sym['cosine_fidelity']:.6f} (> 0.9999)")
print(f"  RMSE                    : {metrics_sym['rmse']:.6f}")
print(f"  Signal-to-Noise Ratio   : {metrics_sym['snr_db']:.2f} dB")
print(f"  Memory Compression      : {metrics_sym['memory_compression_ratio']}x (75% savings)")

# Asymmetric Quantization on bounded activations
activations = rng.uniform(0.05, 5.0, size=(128, 256)).astype(np.float32)
q_asym = CPUQuantizer.quantize_asymmetric_int8(activations)
recon_asym = CPUQuantizer.dequantize(q_asym)
metrics_asym = CPUQuantizer.compute_fidelity_metrics(activations, recon_asym)

print("\nAsymmetric INT8 Quantization Fidelity:")
print(f"  Cosine Fidelity         : {metrics_asym['cosine_fidelity']:.6f}")
print(f"  RMSE                    : {metrics_asym['rmse']:.6f}")
print(f"  Zero Point Offset (Z)   : {q_asym.zero_point}")


### Module 5: The Hybrid RAG + Fine-Tuning Pipeline

In production, the most resilient architecture is often **Hybrid**:
- **Non-Parametric Retriever**: Indexes fresh enterprise documents in ChromaDB with metadata filters.
- **Fine-Tuned Adapter / Reasoning Head**:
  1. Enforces strict citation syntax (`[Citation: Source]`).
  2. Adheres to structured JSON output contracts.
  3. **Calibrated Refusal Gate**: When retrieved context score falls below the confidence threshold, the model executes a deterministic refusal rather than confabulating.

Let us test both factual retrieval with citations and the refusal safety gate.


In [ ]:
corpus = [
    {
        "id": "soc2_sec_4",
        "text": "SOC2 Type II requires 90 days retention for audit telemetry logs before automated tombstoning.",
        "metadata": {"source": "SOC2_Policy_Section_4.2"},
    },
    {
        "id": "arch_sec_9",
        "text": "ChromaDB multi-tenant isolation is enforced via mandatory tenant_id metadata filters.",
        "metadata": {"source": "Architecture_Spec_Section_9.3"},
    },
]

engine = HybridRAGFineTuningEngine(corpus_chunks=corpus)

# Test 1: Grounded Factual Query
query_1 = "How long are telemetry logs retained under SOC2?"
retrieved_1 = engine.retrieve(query_1, top_k=1)
synth_1 = engine.synthesize(query_1, retrieved_1)

print(f"Query 1: {query_1}")
print(f"  Answer     : {synth_1.answer}")
print(f"  Citations  : {synth_1.citations}")
print(f"  Is Refusal : {synth_1.is_refusal}")
print(f"  Latency    : {synth_1.latency_ms:.2f} ms")

# Test 2: Out-of-Domain Query triggering Refusal Gate
query_2 = "What is the quantum encryption frequency of Mars rover communications?"
retrieved_2 = engine.retrieve(query_2, top_k=1)
synth_2 = engine.synthesize(query_2, retrieved_2, confidence_threshold=0.30)

print(f"\nQuery 2: {query_2}")
print(f"  Answer     : {synth_2.answer}")
print(f"  Citations  : {synth_2.citations}")
print(f"  Is Refusal : {synth_2.is_refusal} (Correctly Guarded!)")


### Module 6: Production Serving Contract with FastAPI & ASGI

Production enterprise RAG systems must expose standardized API contracts:
1. `GET /health`: Kubernetes Liveness & Readiness probe returning index status and uptime.
2. `POST /v1/query`: Core query endpoint returning grounded answers, structured citations, retrieved chunks, and `X-Process-Time-Ms` headers.
3. `POST /v1/ingest`: Dynamic ingestion endpoint for adding new document chunks with metadata without server restarts.
4. `GET /v1/metrics`: Prometheus-compatible latency percentiles ($p_{50}, p_{95}, p_{99}$), query counts, and refusal counts.

We test these endpoints live using `starlette.testclient.TestClient`.


In [ ]:
app = create_rag_production_app(service_version="1.0.0")
client = TestClient(app)

# 1. Health Probe Check
res_h = client.get("/health")
print("=== GET /health ===")
print(json.dumps(res_h.json(), indent=2))

# 2. Query Request
query_body = {
    "query": "How is tenant isolation enforced in ChromaDB?",
    "top_k": 2,
    "similarity_threshold": 0.20
}
res_q = client.post("/v1/query", json=query_body)
print("\n=== POST /v1/query ===")
print(f"Status Code      : {res_q.status_code}")
print(f"Process Time Ms  : {res_q.headers.get('X-Process-Time-Ms')}")
print(json.dumps(res_q.json(), indent=2))

# 3. Dynamic Document Ingestion
ingest_body = {
    "documents": [
        {
            "id": "doc_security_patch",
            "text": "Security Patch 2026-B mandates rotating all HMAC keys every 14 days.",
            "metadata": {"source": "SecOps_Advisory_2026B"}
        }
    ]
}
res_i = client.post("/v1/ingest", json=ingest_body)
print("\n=== POST /v1/ingest ===")
print(json.dumps(res_i.json(), indent=2))

# 4. Telemetry Metrics
res_m = client.get("/v1/metrics")
print("\n=== GET /v1/metrics ===")
print(json.dumps(res_m.json(), indent=2))


### Visualizing the Production Serving Topology
We render the end-to-end enterprise deployment topology:
From Client Ingress $\to$ FastAPI Microservice $\to$ Quantized LoRA Head & ChromaDB Storage $\to$ Telemetry & Prometheus Observability.


In [ ]:
topo_path = REPO_ROOT / "assets" / "diagrams" / "track16_production_deployment_architecture.png"
generate_deployment_architecture_diagram(str(topo_path))

img_topo = plt.imread(str(topo_path))
plt.figure(figsize=(15, 7.5))
plt.imshow(img_topo)
plt.axis("off")
plt.title("Enterprise RAG Production Deployment Architecture", fontsize=13, weight="bold", pad=12)
plt.tight_layout()
plt.close()
print(f"Deployment topology diagram loaded from: {topo_path}")


### Module 7: Deployment Readiness Audit & Containerization Hardening

Before deploying a RAG microservice to production Kubernetes clusters, the system configuration must be audited against **8 enterprise operational pillars**:
1. **Unprivileged Non-Root Execution** (UID > 0)
2. **Secret Isolation** (Zero credentials embedded in Docker images)
3. **Liveness & Readiness Probes** (`/health` responding < 100ms)
4. **Bounded Request Timeouts** (Preventing thread exhaustion)
5. **Model Quantization Mode** (INT8 CPU optimization)
6. **Telemetry & Metrics** (Prometheus scraping enabled)
7. **Durable Vector Storage** (Persistent volumes, not ephemeral RAM)
8. **Tenant ACL Enforcement** (Mandatory metadata isolation)

We audit `fixtures/deployment/production_rag_config.json` and generate the hardened multi-stage Dockerfile and Docker Compose specifications.


In [ ]:
config_path = REPO_ROOT / "fixtures" / "deployment" / "production_rag_config.json"
with open(config_path, "r", encoding="utf-8") as f:
    prod_config = json.load(f)

report = DeploymentReadinessAuditor.audit_configuration(prod_config)

print("Deployment Readiness Audit Scorecard:")
print(f"  Total Checks Evaluated : {report.total_checks}")
print(f"  Passed Checks          : {report.passed_checks}")
print(f"  Failed Checks          : {report.failed_checks}")
print(f"  Compliance Score       : {report.compliance_score}%")
print(f"  Production Ready?      : {report.is_ready_for_production}")

print("\nDetailed Check Breakdown:")
for check in report.results:
    mark = "✅ PASS" if check.passed else "❌ FAIL"
    print(f"  {mark} [{check.check_id}] {check.pillar}: {check.description}")

# Inspect generated multi-stage Dockerfile snippet
dockerfile_text = DeploymentReadinessAuditor.generate_dockerfile()
print("\n--- Generated Multi-Stage Dockerfile (First 20 Lines) ---")
for line in dockerfile_text.splitlines()[:20]:
    print(line)


### Module 8: Optional Real LLM Integration (Groq API)
This optional experiment cell queries the live Groq API when `GROQ_API_KEY` is configured in the root `.env`. If no key is present, it executes offline fallback synthesis without failing.


In [ ]:
groq_key = os.getenv("GROQ_API_KEY", "").strip()

if groq_key and not groq_key.startswith("your_"):
    print("GROQ_API_KEY detected. Connecting to official Groq API...")
    try:
        from groq import Groq
        client_groq = Groq(api_key=groq_key)
        prompt = (
            "Compare Parametric vs Non-Parametric memory for enterprise RAG in 2 concise sentences. "
            "Mention the role of INT8 quantization."
        )
        completion = client_groq.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=150,
        )
        print("\nGroq API Response:")
        print(completion.choices[0].message.content)
    except Exception as e:
        print(f"Groq API call encountered an error: {e}. Falling back to offline synthesis.")
else:
    print("GROQ_API_KEY not configured. Running 100% offline-deterministic demonstration:")
    print("  Parametric knowledge encodes patterns into static weights, while non-parametric RAG ")
    print("  dynamically injects verifiable, fresh citations from vector stores into CPU-quantized INT8 runtimes.")


## Academy Milestone Retrospective: Completing all 17 Tracks!

Congratulations! By completing **Track 16**, you have journeyed across the entire spectrum of modern Retrieval-Augmented Generation engineering:

1. **Foundations (Tracks 0–4)**: Tokenization, embeddings, vector similarity math, PyMuPDF chunking, BM25, and end-to-end RAG from first principles in pure Python.
2. **Frameworks & State Graphs (Tracks 5–8)**: LCEL orchestration, LangGraph cyclic state machines, agentic query planning, tool budgets, and working/episodic memory architectures.
3. **Advanced Retrieval Modalities (Tracks 9–13)**: GraphRAG community summaries, Multimodal vision/document embeddings, Google Open Knowledge Format (OKF), TypeSafe Jev decision gates, and Decision Intelligence.
4. **Enterprise & Production Operations (Tracks 14–16)**: Multi-tenancy isolation, prompt injection firewalls, OpenTelemetry distributed tracing, RAG Triad automated evaluation, semantic drift detection, LoRA fine-tuning trade-offs, INT8 CPU quantization, and FastAPI Kubernetes serving.

You possess the theoretical understanding and hands-on code required to architect, implement, evaluate, and deploy production-grade RAG systems.
